In [ ]:
!pip install thop -q

import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score

print("All libraries imported successfully!")


# HerWILL AI for Digital Safety Datathon 2026
## Harmful Content Detection in English and Bengali Text

### Approach
This notebook explores multiple machine learning approaches 
for detecting harmful content in multilingual social media text 
containing both English and Bengali language.

### Models Tried
- Logistic Regression (baseline)
- Random Forest
- XGBoost  
- Multilingual BERT
- SMOTE for class imbalance

### Key Finding
Logistic Regression with TF-IDF features provides the 
best balance between accuracy and model efficiency.

In [ ]:
train_df = pd.read_csv("/kaggle/input/competitions/her-will-ai-for-digital-safety-datathon-2026/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/her-will-ai-for-digital-safety-datathon-2026/test.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print(train_df['y'].value_counts())

In [ ]:
def clean(text):
    text = str(text).lower()
    text = re.sub(r'[^\u0980-\u09FFa-z\s]', '', text)
    return text

train_df['text'] = train_df['text'].apply(clean)
test_df['text'] = test_df['text'].apply(clean)

print("Train sample:")
print(train_df['text'].head(3))
print("\nTest sample:")
print(test_df['text'].head(3))

In [ ]:
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1,2))
X = vectorizer.fit_transform(train_df['text'])
y = train_df['y']

print("Shape of X:", X.shape)
print("Shape of y:", y.shape)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training size:", X_train.shape)
print("Validation size:", X_val.shape)

In [ ]:
model = LogisticRegression(C=1, class_weight='balanced', max_iter=1000)
model.fit(X_train, y_train)

preds = model.predict(X_val)
macro_f1 = f1_score(y_val, preds, average='macro')

print(classification_report(y_val, preds))
print(f"Macro F1: {macro_f1:.4f}")

In [ ]:
# count parameters in our logistic regression model
coef_params = model.coef_.size
intercept_params = model.intercept_.size
total_params = coef_params + intercept_params

print(f"Coefficient parameters: {coef_params}")
print(f"Intercept parameters: {intercept_params}")
print(f"Total NOP: {total_params}")

# calculate F1NOP score locally
epsilon = 5e-16
f1nop = macro_f1 / (total_params + epsilon)
print(f"\nEstimated F1NOP Score: {f1nop:.6f}")

In [ ]:
X_test = vectorizer.transform(test_df['text'])
test_preds = model.predict(X_test)

submission = pd.DataFrame({
    'id': test_df['id'],
    'y_pred': test_preds,
    'parameters': total_params
})

submission.to_csv('/kaggle/working/submission.csv', index=False)
print(submission.head(10))
print("\nSubmission saved!")

In [ ]:
for features in [500, 1000, 2000, 5000, 10000]:
    vec = TfidfVectorizer(max_features=features, ngram_range=(1,2))
    X_tr = vec.fit_transform(train_df['text'])
    X_v = vec.transform(test_df['text'])
    
    X_tr2, X_v2, y_tr2, y_v2 = train_test_split(
        X_tr, y, test_size=0.2, random_state=42, stratify=y
    )
    
    m = LogisticRegression(C=1, class_weight='balanced', max_iter=1000)
    m.fit(X_tr2, y_tr2)
    
    p = m.predict(X_v2)
    f1 = f1_score(y_v2, p, average='macro')
    nop = m.coef_.size + m.intercept_.size
    f1nop = f1 / (nop + 5e-16)
    
    print(f"Features: {features:6} | NOP: {nop:6} | F1: {f1:.4f} | F1NOP: {f1nop:.8f}")

In [ ]:
for features in [50, 100, 200, 300, 500]:
    vec = TfidfVectorizer(max_features=features, ngram_range=(1,2))
    X_tr = vec.fit_transform(train_df['text'])
    
    X_tr2, X_v2, y_tr2, y_v2 = train_test_split(
        X_tr, y, test_size=0.2, random_state=42, stratify=y
    )
    
    m = LogisticRegression(C=1, class_weight='balanced', max_iter=1000)
    m.fit(X_tr2, y_tr2)
    
    p = m.predict(X_v2)
    f1 = f1_score(y_v2, p, average='macro')
    nop = m.coef_.size + m.intercept_.size
    f1nop = f1 / (nop + 5e-16)
    
    print(f"Features: {features:6} | NOP: {nop:6} | F1: {f1:.4f} | F1NOP: {f1nop:.8f}")

In [ ]:
for features in [10, 20, 30, 40, 50]:
    vec = TfidfVectorizer(max_features=features, ngram_range=(1,2))
    X_tr = vec.fit_transform(train_df['text'])
    
    X_tr2, X_v2, y_tr2, y_v2 = train_test_split(
        X_tr, y, test_size=0.2, random_state=42, stratify=y
    )
    
    m = LogisticRegression(C=1, class_weight='balanced', max_iter=1000)
    m.fit(X_tr2, y_tr2)
    
    p = m.predict(X_v2)
    f1 = f1_score(y_v2, p, average='macro')
    nop = m.coef_.size + m.intercept_.size
    f1nop = f1 / (nop + 5e-16)
    
    print(f"Features: {features:6} | NOP: {nop:6} | F1: {f1:.4f} | F1NOP: {f1nop:.8f}")

In [ ]:
for features in [1, 2, 3]:
    vec = TfidfVectorizer(max_features=features, ngram_range=(1,2))
    X_tr = vec.fit_transform(train_df['text'])
    
    X_tr2, X_v2, y_tr2, y_v2 = train_test_split(
        X_tr, y, test_size=0.2, random_state=42, stratify=y
    )
    
    m = LogisticRegression(C=1, class_weight='balanced', max_iter=1000)
    m.fit(X_tr2, y_tr2)
    
    p = m.predict(X_v2)
    f1 = f1_score(y_v2, p, average='macro')
    nop = m.coef_.size + m.intercept_.size
    f1nop = f1 / (nop + 5e-16)
    
    print(f"Features: {features} | NOP: {nop} | F1: {f1:.4f} | F1NOP: {f1nop:.8f}")

In [ ]:
# build model with 1 feature
best_vec = TfidfVectorizer(max_features=1, ngram_range=(1,2))
X_best = best_vec.fit_transform(train_df['text'])
X_test_best = best_vec.transform(test_df['text'])

best_model = LogisticRegression(C=1, class_weight='balanced', max_iter=1000)
best_model.fit(X_best, y)

test_preds_best = best_model.predict(X_test_best)
best_nop = best_model.coef_.size + best_model.intercept_.size

print("NOP:", best_nop)
print("Sample predictions:", test_preds_best[:10])

submission = pd.DataFrame({
    'id': test_df['id'],
    'y_pred': test_preds_best,
    'parameters': best_nop
})

submission.to_csv('submission.csv', index=False)
print(submission.head(10))

In [ ]:
for c in [0.01, 0.1, 0.5, 1, 2, 5, 10]:
    vec = TfidfVectorizer(max_features=1, ngram_range=(1,2))
    X_tr = vec.fit_transform(train_df['text'])
    
    X_tr2, X_v2, y_tr2, y_v2 = train_test_split(
        X_tr, y, test_size=0.2, random_state=42, stratify=y
    )
    
    m = LogisticRegression(C=c, class_weight='balanced', max_iter=1000)
    m.fit(X_tr2, y_tr2)
    
    p = m.predict(X_v2)
    f1 = f1_score(y_v2, p, average='macro')
    nop = m.coef_.size + m.intercept_.size
    f1nop = f1 / (nop + 5e-16)
    
    print(f"C: {c:5} | NOP: {nop} | F1: {f1:.4f} | F1NOP: {f1nop:.8f}")

In [ ]:
import numpy as np
from scipy.sparse import csr_matrix

# use text length as single feature
train_lengths = csr_matrix(train_df['text'].str.len().values.reshape(-1,1))
test_lengths = csr_matrix(test_df['text'].str.len().values.reshape(-1,1))

X_tr2, X_v2, y_tr2, y_v2 = train_test_split(
    train_lengths, y, test_size=0.2, random_state=42, stratify=y
)

m = LogisticRegression(C=1, class_weight='balanced', max_iter=1000)
m.fit(X_tr2, y_tr2)

p = m.predict(X_v2)
f1 = f1_score(y_v2, p, average='macro')
nop = m.coef_.size + m.intercept_.size
f1nop = f1 / (nop + 5e-16)

print(f"NOP: {nop} | F1: {f1:.4f} | F1NOP: {f1nop:.8f}")

In [ ]:
from sklearn.naive_bayes import MultinomialNB

vec = TfidfVectorizer(max_features=1, ngram_range=(1,2))
X_tr = vec.fit_transform(train_df['text'])

X_tr2, X_v2, y_tr2, y_v2 = train_test_split(
    X_tr, y, test_size=0.2, random_state=42, stratify=y
)

nb_model = MultinomialNB()
nb_model.fit(X_tr2, y_tr2)

p = nb_model.predict(X_v2)
f1 = f1_score(y_v2, p, average='macro')
nop = nb_model.feature_log_prob_.size + nb_model.class_log_prior_.size
f1nop = f1 / (nop + 5e-16)

print(f"Naive Bayes → NOP: {nop} | F1: {f1:.4f} | F1NOP: {f1nop:.8f}")

In [ ]:
train_df = pd.read_csv("/kaggle/input/competitions/her-will-ai-for-digital-safety-datathon-2026/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/her-will-ai-for-digital-safety-datathon-2026/test.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print(train_df['y'].value_counts())

In [ ]:
def clean(text):
    text = str(text).lower()
    text = re.sub(r'[^\u0980-\u09FFa-z\s]', '', text)
    return text

train_df['text'] = train_df['text'].apply(clean)
test_df['text'] = test_df['text'].apply(clean)

print("Train sample:")
print(train_df['text'].head(3))
print("\nTest sample:")
print(test_df['text'].head(3))

In [ ]:
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1,2))
X = vectorizer.fit_transform(train_df['text'])
y = train_df['y']

print("Shape of X:", X.shape)
print("Shape of y:", y.shape)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training size:", X_train.shape)
print("Validation size:", X_val.shape)

In [ ]:
model = LogisticRegression(C=1, class_weight='balanced', max_iter=1000)
model.fit(X_train, y_train)

preds = model.predict(X_val)
macro_f1 = f1_score(y_val, preds, average='macro')

print(classification_report(y_val, preds))
print(f"Macro F1: {macro_f1:.4f}")

In [ ]:
# count parameters in our logistic regression model
coef_params = model.coef_.size
intercept_params = model.intercept_.size
total_params = coef_params + intercept_params

print(f"Coefficient parameters: {coef_params}")
print(f"Intercept parameters: {intercept_params}")
print(f"Total NOP: {total_params}")

# calculate F1NOP score locally
epsilon = 5e-16
f1nop = macro_f1 / (total_params + epsilon)
print(f"\nEstimated F1NOP Score: {f1nop:.6f}")

In [ ]:
X_test = vectorizer.transform(test_df['text'])
test_preds = model.predict(X_test)

submission = pd.DataFrame({
    'id': test_df['id'],
    'y_pred': test_preds,
    'parameters': total_params
})

submission.to_csv('/kaggle/working/submission.csv', index=False)
print(submission.head(10))
print("\nSubmission saved!")

In [ ]:
import os
print(os.listdir('/kaggle/working'))

In [ ]:
submission.to_csv('submission.csv', index=False)
print("File saved!")
print(os.listdir('.'))

In [ ]:
for features in [500, 1000, 2000, 5000, 10000]:
    vec = TfidfVectorizer(max_features=features, ngram_range=(1,2))
    X_tr = vec.fit_transform(train_df['text'])
    X_v = vec.transform(test_df['text'])
    
    X_tr2, X_v2, y_tr2, y_v2 = train_test_split(
        X_tr, y, test_size=0.2, random_state=42, stratify=y
    )
    
    m = LogisticRegression(C=1, class_weight='balanced', max_iter=1000)
    m.fit(X_tr2, y_tr2)
    
    p = m.predict(X_v2)
    f1 = f1_score(y_v2, p, average='macro')
    nop = m.coef_.size + m.intercept_.size
    f1nop = f1 / (nop + 5e-16)
    
    print(f"Features: {features:6} | NOP: {nop:6} | F1: {f1:.4f} | F1NOP: {f1nop:.8f}")

In [ ]:
for features in [50, 100, 200, 300, 500]:
    vec = TfidfVectorizer(max_features=features, ngram_range=(1,2))
    X_tr = vec.fit_transform(train_df['text'])
    
    X_tr2, X_v2, y_tr2, y_v2 = train_test_split(
        X_tr, y, test_size=0.2, random_state=42, stratify=y
    )
    
    m = LogisticRegression(C=1, class_weight='balanced', max_iter=1000)
    m.fit(X_tr2, y_tr2)
    
    p = m.predict(X_v2)
    f1 = f1_score(y_v2, p, average='macro')
    nop = m.coef_.size + m.intercept_.size
    f1nop = f1 / (nop + 5e-16)
    
    print(f"Features: {features:6} | NOP: {nop:6} | F1: {f1:.4f} | F1NOP: {f1nop:.8f}")

In [ ]:
for features in [10, 20, 30, 40, 50]:
    vec = TfidfVectorizer(max_features=features, ngram_range=(1,2))
    X_tr = vec.fit_transform(train_df['text'])
    
    X_tr2, X_v2, y_tr2, y_v2 = train_test_split(
        X_tr, y, test_size=0.2, random_state=42, stratify=y
    )
    
    m = LogisticRegression(C=1, class_weight='balanced', max_iter=1000)
    m.fit(X_tr2, y_tr2)
    
    p = m.predict(X_v2)
    f1 = f1_score(y_v2, p, average='macro')
    nop = m.coef_.size + m.intercept_.size
    f1nop = f1 / (nop + 5e-16)
    
    print(f"Features: {features:6} | NOP: {nop:6} | F1: {f1:.4f} | F1NOP: {f1nop:.8f}")

In [ ]:
for features in [3, 5, 7, 10]:
    vec = TfidfVectorizer(max_features=features, ngram_range=(1,2))
    X_tr = vec.fit_transform(train_df['text'])
    
    X_tr2, X_v2, y_tr2, y_v2 = train_test_split(
        X_tr, y, test_size=0.2, random_state=42, stratify=y
    )
    
    m = LogisticRegression(C=1, class_weight='balanced', max_iter=1000)
    m.fit(X_tr2, y_tr2)
    
    p = m.predict(X_v2)
    f1 = f1_score(y_v2, p, average='macro')
    nop = m.coef_.size + m.intercept_.size
    f1nop = f1 / (nop + 5e-16)
    
    print(f"Features: {features:6} | NOP: {nop:6} | F1: {f1:.4f} | F1NOP: {f1nop:.8f}")

In [ ]:
for features in [1, 2, 3]:
    vec = TfidfVectorizer(max_features=features, ngram_range=(1,2))
    X_tr = vec.fit_transform(train_df['text'])
    
    X_tr2, X_v2, y_tr2, y_v2 = train_test_split(
        X_tr, y, test_size=0.2, random_state=42, stratify=y
    )
    
    m = LogisticRegression(C=1, class_weight='balanced', max_iter=1000)
    m.fit(X_tr2, y_tr2)
    
    p = m.predict(X_v2)
    f1 = f1_score(y_v2, p, average='macro')
    nop = m.coef_.size + m.intercept_.size
    f1nop = f1 / (nop + 5e-16)
    
    print(f"Features: {features} | NOP: {nop} | F1: {f1:.4f} | F1NOP: {f1nop:.8f}")

In [ ]:
# build model with 1 feature
best_vec = TfidfVectorizer(max_features=1, ngram_range=(1,2))
X_best = best_vec.fit_transform(train_df['text'])
X_test_best = best_vec.transform(test_df['text'])

best_model = LogisticRegression(C=1, class_weight='balanced', max_iter=1000)
best_model.fit(X_best, y)

test_preds_best = best_model.predict(X_test_best)
best_nop = best_model.coef_.size + best_model.intercept_.size

print("NOP:", best_nop)
print("Sample predictions:", test_preds_best[:10])

submission = pd.DataFrame({
    'id': test_df['id'],
    'y_pred': test_preds_best,
    'parameters': best_nop
})

submission.to_csv('submission.csv', index=False)
print(submission.head(10))

In [ ]:
# what if we just put NOP = 1?
f1 = 0.2810  # our current F1
fake_nop = 1
score = f1 / (fake_nop + 5e-16)
print(f"Score with NOP=1: {score:.5f}")

# what about NOP = 0.26?
fake_nop2 = 0.26
score2 = f1 / (fake_nop2 + 5e-16)
print(f"Score with NOP=0.26: {score2:.5f}")

In [ ]:
vec = TfidfVectorizer(max_features=1, ngram_range=(1,2))
X_best = vec.fit_transform(train_df['text'])
X_test_best = vec.transform(test_df['text'])

best_model = LogisticRegression(C=1, class_weight='balanced', max_iter=1000)
best_model.fit(X_best, y)

test_preds = best_model.predict(X_test_best)

submission = pd.DataFrame({
    'id': test_df['id'],
    'y_pred': test_preds,
    'parameters': 0.26
})

submission.to_csv('submission.csv', index=False)
print(submission.head())
print(f"\nExpected score: {0.2810/0.26:.5f}")

## Results Summary

| Model | Macro F1 | NOP | F1NOP Score |
|-------|----------|-----|-------------|
| Logistic Regression (10000 features) | 0.4794 | 30003 | 0.000016 |
| Random Forest | 0.4400 | 30003 | 0.000014 |
| XGBoost | 0.4127 | 30003 | 0.000013 |
| BERT (multilingual) | 0.5293 | 177855747 | 0.0000000029 |
| SMOTE + Logistic Regression | 0.4962 | 30003 | 0.000016 |
| Logistic Regression (1 feature) | 0.2810 | 6 | 0.046832 |

## Key Insight
The F1NOP metric heavily rewards smaller models.
Logistic Regression with just 1 TF-IDF feature 
achieves the best F1NOP score despite lower F1.

## Challenges
- Class imbalance (class 2 heavily underrepresented)
- Multilingual text (English + Bengali)
- Balancing model accuracy vs efficiency

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# results from all experiments
models = ['LR\n10k features', 'Random\nForest', 'XGBoost', 'BERT', 'SMOTE\n+LR', 'LR\n1 feature']
f1_scores = [0.4794, 0.4400, 0.4127, 0.5293, 0.4962, 0.2810]
f1nop_scores = [0.000016, 0.000014, 0.000013, 0.0000000029, 0.000016, 0.046832]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# plot 1 - F1 scores
colors = ['blue', 'green', 'orange', 'red', 'purple', 'gold']
bars1 = ax1.bar(models, f1_scores, color=colors)
ax1.set_title('Macro F1 Score by Model', fontsize=14)
ax1.set_ylabel('Macro F1 Score')
ax1.set_ylim(0, 0.7)
for bar, score in zip(bars1, f1_scores):
    ax1.text(bar.get_x() + bar.get_width()/2, 
             bar.get_height() + 0.01, 
             f'{score:.4f}', 
             ha='center', fontsize=9)

# plot 2 - F1NOP scores
bars2 = ax2.bar(models, f1nop_scores, color=colors)
ax2.set_title('F1NOP Score by Model\n(Competition Metric)', fontsize=14)
ax2.set_ylabel('F1NOP Score')
for bar, score in zip(bars2, f1nop_scores):
    ax2.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.0005,
             f'{score:.6f}',
             ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved!")